# 1.获取大模型

In [ ]:
# 导入 dotenv 库的 load_dotenv 函数，用于加载环境变量配置文件 (.env) 中的配置
import dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
import os

dotenv.load_dotenv()  # 加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = os.getenv('OPENAI_BASE_URL')

# 创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")  # 默认使用 gpt-3.5-turbo

# 直接提供问题，并调用 LLM
response = llm.invoke("什么是大模型？")
print(response)


# 2.使用提示次模板

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 使用提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者"),
    ("user", "{input}")
])

# 创建链
chain = prompt | llm

# 调用链
response = chain.invoke({"input": "人工智能"})
print(response)


# 3.使用输出解析器

In [ ]:
# 导入 dotenv 库的 load_dotenv 函数，用于加载环境变量配置文件 (.env) 中的配置
import dotenv
from langchain_openai import ChatOpenAI
import os

dotenv.load_dotenv()  # 加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = os.getenv('OPENAI_BASE_URL')

# 创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")  # 默认使用 gpt-3.5-turbo


# 使用提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者"),
    ("user", "{input}")
])

# 创建outputparser
output_parser = JsonOutputParser()


# 创建链
chain = prompt | llm |output_parser


# 调用链
response = chain.invoke({"input": "人工智能"})
print(response)


4.使用向量存储

In [17]:
# 导入 dotenv 库的 load_dotenv 函数，用于加载环境变量配置文件 (.env) 中的配置
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import bs4

# 使用 WebBaseLoader 加载网页内容
loader = WebBaseLoader(
    web_path="https://www.gov.cn/yaowen/liebiao/202603/content_7063789.htm",
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="UCAP-CONTENT"))
)
docs = loader.load()
print(f"加载的文档数量：{len(docs)}")
if len(docs) > 0:
    print(f"第一个文档内容长度：{len(docs[0].page_content)}")


# 三种嵌入模型： 1.通过 API 调用商业模型  2.本地运行开源模型 3.针对特定硬件的优化模型
# Hugging Face 的 sentence-transformers：HuggingFaceEmbeddings。这是本地使用的首选，有海量的中文和英文模型可供选择，如 BAAI/bge-large-zh-v1.5（中文）或 all-MiniLM-L6-v2（英文）。
#
# Ollama：OllamaEmbeddings。通过 Ollama 这个易用的本地模型管理工具，可以一键运行 nomic-embed-text 等嵌入模型。
#
# Llama.cpp：LlamaCppEmbeddings。适合在资源受限的环境下运行量化后的模型

# 对于嵌入模型，这里通过 API 调用
# 3. 向量化——使用专业嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5"
)


# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
print(f"分割后的文档片段数量：{len(documents)}")

if len(documents) == 0:
    raise ValueError("没有可用的文档片段，请检查网页是否可访问或 HTML 元素是否存在")

# 向量存储 embeddings 会将 documents 中的每个文本片段转换为向量，并将这些向量存储在 FAISS 向量数据库中
vector = FAISS.from_documents(documents, embeddings)

加载的文档数量：1
第一个文档内容长度：3655


/var/folders/jp/vzmq7qb56snc4l68t853gtzr0000gn/T/ipykernel_65297/3784145277.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/Users/sunshoucai/PycharmProjects/agent-learning/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 71/71 [00:00<00:00, 12557.80it/s]
BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | 

分割后的文档片段数量：9


# 5.RAG检索增强生成

In [ ]:
# 导入 dotenv 库的 load_dotenv 函数，用于加载环境变量配置文件 (.env) 中的配置
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
import bs4
import dotenv

dotenv.load_dotenv()

# 使用 WebBaseLoader 加载网页内容
loader = WebBaseLoader(
    web_path="https://www.gov.cn/yaowen/liebiao/202603/content_7063789.htm",
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="UCAP-CONTENT"))
)
docs = loader.load()
print(f"加载的文档数量：{len(docs)}")
if len(docs) > 0:
    print(f"第一个文档内容长度：{len(docs[0].page_content)}")

# 对于嵌入模型，这里通过 API 调用
# 3. 向量化——使用专业嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5"
)


# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
print(f"分割后的文档片段数量：{len(documents)}")

if len(documents) == 0:
    raise ValueError("没有可用的文档片段，请检查网页是否可访问或 HTML 元素是否存在")

# 向量存储 embeddings 会将 documents 中的每个文本片段转换为向量，并将这些向量存储在 FAISS 向量数据库中
vector = FAISS.from_documents(documents, embeddings)


from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# 4. 检索器设置
retriever = vector.as_retriever()
retriever.search_kwargs = {"k": 3}
docs = retriever.invoke("护理保险制度是什么？")

# 打印检索结果
for i, doc in enumerate(docs):
    print(f"⭐ 第{i+1}条规定：")
    print(doc)

# 5. 定义提示词模板

prompt_template = """
你是一个回答机器人。
你的任务是根据下述给定的已知信息回答用户问题。
确保你的回复完全依据下述已知信息，不要编造答案。
如果下述已知信息不足以回答用户的问题，请直接回复"我无法回答您的问题"。

已知信息：
{info}

用户问：
{question}
"""

prompt = PromptTemplate.from_template(prompt_template)

# 6. 创建 RAG 链

# 创建格式化函数，将检索结果转换为字符串
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")  # 默认使用 gpt-3.5-turbo

# 构建 RAG 链
rag_chain = (
    {"info": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 7. 测试 RAG 检索增强生成
response = rag_chain.invoke("护理保险制度是什么？")
print("\n🤖 RAG 回答：")
print(response)

# 6.使用agent

In [ ]:
# 导入必要的库
from langchain_classic.agents import AgentExecutor, create_openai_tools_agent
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
import bs4
import dotenv

dotenv.load_dotenv()

print("=" * 60)
print("🤖 LangChain Agent 示例 - 政策咨询助手")
print("=" * 60)

# ==================== 1. 准备知识库 ====================
print("\n📚 步骤 1: 加载和准备知识库...")

# 使用 WebBaseLoader 加载网页内容
loader = WebBaseLoader(
    web_path="https://www.gov.cn/yaowen/liebiao/202603/content_7063789.htm",
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="UCAP-CONTENT"))
)
docs = loader.load()
print(f"✓ 加载的文档数量：{len(docs)}")
if len(docs) > 0:
    print(f"✓ 第一个文档内容长度：{len(docs[0].page_content)}")

# 向量化——使用专业嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5"
)

# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
print(f"✓ 分割后的文档片段数量：{len(documents)}")

if len(documents) == 0:
    raise ValueError("没有可用的文档片段，请检查网页是否可访问或 HTML 元素是否存在")

# 向量存储
vector = FAISS.from_documents(documents, embeddings)
print("✓ FAISS 向量库创建成功！")

# ==================== 2. 创建检索工具 ====================
print("\n🛠️  步骤 2: 创建 Agent 工具...")

# 创建检索器
retriever = vector.as_retriever(search_kwargs={"k": 3})


# 定义检索工具
@tool
def search_policy(query: str) -> str:
    """搜索相关政策信息。

    当你需要回答关于政策、法规、制度的问题时使用此工具。

    Args:
        query: 用户的问题或查询关键词
    """
    print(f"  🔍 Agent 正在搜索：{query}")
    results = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in results)


# 创建工具列表
tools = [search_policy]

# ==================== 3. 创建大模型和 Agent ====================
print("\n 步骤 3: 创建 Agent...")

# 创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")

# 定义系统提示词
system_prompt = """你是一个专业的政策咨询助手。你可以使用提供的工具来回答用户关于政策的问题。

请遵循以下原则：
1. 首先理解用户的问题
2. 如果需要查找信息，使用 search_policy 工具搜索相关政策
3. 根据搜索结果给出准确、完整的回答
4. 如果搜索结果为空或不足以回答问题，诚实地告诉用户
5. 回答要简洁明了，避免冗长

用户问题:
{input}

{agent_scratchpad}
"""

# 创建提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 创建工具调用 Agent
agent = create_openai_tools_agent(llm, tools, prompt)

# 创建 Agent 执行器
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3,
)

print("✓ Agent 创建成功！")

# ==================== 4. 测试 Agent ====================
print("\n💬 步骤 4: 测试 Agent...")
print("-" * 60)

# 测试问题
test_questions = [
    "护理保险制度是什么？",
    "这个制度适用于哪些人群？",
]

for question in test_questions:
    print(f"\n❓ 问题：{question}")
    print("-" * 60)

    try:
        # 调用 Agent
        response = agent_executor.invoke({"input": question})

        print(f"\n✅ 回答：{response['output']}")
        print("-" * 60)
    except Exception as e:
        print(f"\n❌ 回答失败：{str(e)}")
        print("-" * 60)

# ==================== 5. 交互式对话 ====================
print("\n🎯 现在进入交互式对话模式（输入 'quit' 退出）")
print("=" * 60)

while True:
    try:
        user_input = input("\n👤 你：").strip()

        if user_input.lower() in ['quit', 'exit', '退出']:
            print("👋 再见！")
            break

        if not user_input:
            continue

        print("-" * 60)
        response = agent_executor.invoke({"input": user_input})
        print(f"\n🤖 Agent: {response['output']}")

    except KeyboardInterrupt:
        print("\n👋 再见！")
        break
    except Exception as e:
        print(f"\n❌ 发生错误：{str(e)}")
